# 01 - Fundamentos del Algoritmo KNN y Métricas de Distancia

## Curso de Machine Learning: Modelos de Clasificación

El algoritmo **K-Nearest Neighbors (K-Vecinos Más Cercanos - KNN)** es uno de los métodos supervisados más intuitivos, elegantes y efectivos del aprendizaje automático. A diferencia de los modelos paramétricos como la regresión lineal o logística, KNN pertenece a la familia de modelos **no paramétricos basados en instancias (Instance-Based Learning / Lazy Learning)**.

---

### Contenidos
1. Paradigma de Aprendizaje Basado en Instancias (Lazy vs. Eager Learning)
2. Geometría y Métricas de Distancia: Euclidiana, Manhattan, Minkowski y Coseno
3. Visualización Geométrica de las Normas L1, L2 y L-infinito
4. El Rol Crítico del Escalado de Características (Feature Scaling)
5. Demostración Práctica con Dataset Cáncer de Mama: Impacto de la Normalización
6. Implementación Robusta con Scikit-Learn Pipelines
7. Conclusiones y Puntos Clave


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer, load_iris
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# Configuracion de estilo visual
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)
print("Bibliotecas importadas y entorno configurado correctamente.")


## 1. Paradigma de Aprendizaje Basado en Instancias (Lazy vs. Eager Learning)

En los algoritmos convencionales de Machine Learning (**Eager Learning**, como Regresión Logística, Árboles de Decisión o Redes Neuronales), la fase de entrenamiento implica estimar parámetros fijos (pesos $\mathbf{w}$, sesgos $b$ o divisiones de umbrales) a partir de una función de costo. Una vez finalizado el entrenamiento, los datos de entrenamiento pueden descartarse.

En contraste, en **KNN (Lazy Learning)**:
- **Fase de entrenamiento ($O(1)$)**: Consiste simplemente en almacenar el conjunto de datos de entrenamiento en memoria. No se realiza ninguna generalización explícita previa.
- **Fase de predicción ($O(N \cdot D)$)**: Para clasificar una nueva consulta $\mathbf{x}^*$:
  1. Se calcula la distancia matemática entre $\mathbf{x}^*$ y cada una de las $N$ muestras almacenadas en el espacio de dimensión $D$.
  2. Se seleccionan los $K$ puntos más cercanos (vecinos).
  3. Se realiza una votación mayoritaria (o ponderada) entre las etiquetas de los $K$ vecinos para asignar la clase a $\mathbf{x}^*$.


## 2. Geometría y Métricas de Distancia

La noción de "similitud" en KNN se cuantifica matemáticamente a través de funciones de distancia entre dos vectores $\mathbf{u}, \mathbf{v} \in \mathbb{R}^D$:

### 1. Distancia Euclidiana (Norma $L_2$)
Es la distancia euclidiana en línea recta en el espacio euclidiano:
$$d_2(\mathbf{u}, \mathbf{v}) = \sqrt{\sum_{i=1}^D (u_i - v_i)^2}$$

### 2. Distancia Manhattan (Norma $L_1$ / Taxicab)
Mide la suma de las diferencias absolutas de sus coordenadas (como desplazarse en una cuadrícula urbana):
$$d_1(\mathbf{u}, \mathbf{v}) = \sum_{i=1}^D |u_i - v_i|$$

### 3. Distancia Minkowski (Norma general $L_p$)
Generalización paramétrica que unifica las normas anteriores según el parámetro $p \ge 1$:
$$d_p(\mathbf{u}, \mathbf{v}) = \left( \sum_{i=1}^D |u_i - v_i|^p \right)^{1/p}$$
- Si $p = 1$: Equivale a la distancia Manhattan.
- Si $p = 2$: Equivale a la distancia Euclidiana.
- Si $p \to \infty$: Converge a la distancia Chebyshev ($L_\infty = \max_i |u_i - v_i|$).

### 4. Distancia Coseno
Mide el ángulo entre dos vectores independientemente de su magnitud (clave en procesamiento de texto y embeddings):
$$d_{\cos}(\mathbf{u}, \mathbf{v}) = 1 - \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\|_2 \|\mathbf{v}\|_2}$$


In [ ]:
# Comparativa de Formas Geometricas: Bolas Unitarias en R^2
# Puntos que satisfacen ||x||_p <= 1 para diferentes normas

theta = np.linspace(0, 2*np.pi, 1000)
circle_x, circle_y = np.cos(theta), np.sin(theta)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# L1 (Manhattan): Rombo
t = np.linspace(-1, 1, 500)
axes[0].fill(np.concatenate([t, t[::-1]]), np.concatenate([1 - np.abs(t), -(1 - np.abs(t))]), color='#2b5c8f', alpha=0.3)
axes[0].plot(np.concatenate([t, t[::-1]]), np.concatenate([1 - np.abs(t), -(1 - np.abs(t))]), color='#2b5c8f', lw=2)
axes[0].set_title(r'Norma $L_1$ (Manhattan): $|x| + |y| \leq 1$', fontsize=11, fontweight='bold')

# L2 (Euclidiana): Circulo
axes[1].fill(circle_x, circle_y, color='#2ca02c', alpha=0.3)
axes[1].plot(circle_x, circle_y, color='#2ca02c', lw=2)
axes[1].set_title(r'Norma $L_2$ (Euclidiana): $\sqrt{x^2 + y^2} \leq 1$', fontsize=11, fontweight='bold')

# L_infinito (Chebyshev): Cuadrado
square_x = [-1, 1, 1, -1, -1]
square_y = [-1, -1, 1, 1, -1]
axes[2].fill(square_x, square_y, color='#d62728', alpha=0.3)
axes[2].plot(square_x, square_y, color='#d62728', lw=2)
axes[2].set_title(r'Norma $L_\infty$ (Chebyshev): $\max(|x|, |y|) \leq 1$', fontsize=11, fontweight='bold')

for ax in axes:
    ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
    ax.axvline(0, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)
    ax.set_aspect('equal')
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')

plt.suptitle('Geometria de Bolas Unitarias para Distintas Normas de Minkowski', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 3. El Rol Crítico del Escalado de Características (Feature Scaling)

Dado que KNN computa distancias numéricas directas entre coordenadas, **es extremadamente sensible a la escala de las variables**:
- Supongamos dos variables: *Ingreso Anual* (rango $\$10,000$ a $\$200,000$) y *Edad* (rango $18$ a $85$ años).
- La diferencia de $\$1,000$ en ingreso aportará $(1000)^2 = 1,000,000$ al cálculo de la distancia euclidiana, mientras que una diferencia de $10$ años aportará $(10)^2 = 100$.
- **Consecuencia**: La variable con magnitudes numéricas superiores dominará el $99.99\%$ de la distancia euclidiana, haciendo que la edad sea matemáticamente irrelevante.

### Métodos Principales de Escalado:
1. **Estandarización Z-score (`StandardScaler`)**:
   $$z = \frac{x - \mu}{\sigma} \implies \mu=0, \sigma=1$$
2. **Normalización Min-Max (`MinMaxScaler`)**:
   $$x_{norm} = \frac{x - x_{min}}{x_{max} - x_{min}} \implies x_{norm} \in [0, 1]$$


In [ ]:
# Demostracion empirica: KNN con vs sin escalado en el Dataset Cancer de Mama
cancer = load_breast_cancer()
X = cancer.data
y = cancer.target

# Division estratificada entrenamiento / prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"Dimensiones de entrenamiento: {X_train.shape}")
print(f"Rango caracteristica 0 (mean radius): [{X_train[:, 0].min():.2f}, {X_train[:, 0].max():.2f}]")
print(f"Rango caracteristica 3 (mean area):   [{X_train[:, 3].min():.2f}, {X_train[:, 3].max():.2f}]")


In [ ]:
# Experimento 1: KNN sin escalado
knn_raw = KNeighborsClassifier(n_neighbors=5)
knn_raw.fit(X_train, y_train)
y_pred_raw = knn_raw.predict(X_test)
acc_raw = accuracy_score(y_test, y_pred_raw)

# Experimento 2: KNN con StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn_scaled = KNeighborsClassifier(n_neighbors=5)
knn_scaled.fit(X_train_scaled, y_train)
y_pred_scaled = knn_scaled.predict(X_test_scaled)
acc_scaled = accuracy_score(y_test, y_pred_scaled)

print("="*55)
print(f"Exactitud (Accuracy) KNN SIN Escalado: {acc_raw * 100:.2f}%")
print(f"Exactitud (Accuracy) KNN CON StandardScaler: {acc_scaled * 100:.2f}%")
print(f"Diferencia a favor del escalado: +{(acc_scaled - acc_raw)*100:.2f}% puntos porcentuales")
print("="*55)


In [ ]:
# Visualizacion de Matrices de Confusion comparativas
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_raw, 
    display_labels=cancer.target_names, 
    cmap='Blues', 
    ax=axes[0]
)
axes[0].set_title(f'Sin Escalado (Acc: {acc_raw*100:.1f}%)', fontsize=11, fontweight='bold')
axes[0].grid(False)

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_scaled, 
    display_labels=cancer.target_names, 
    cmap='Greens', 
    ax=axes[1]
)
axes[1].set_title(f'Con StandardScaler (Acc: {acc_scaled*100:.1f}%)', fontsize=11, fontweight='bold')
axes[1].grid(False)

plt.suptitle('Comparativa del Desempeño de KNN ante el Preprocesamiento de Escala', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 4. Implementación Robusta con Scikit-Learn Pipelines

Para prevenir la **fuga de información (Data Leakage)** al trabajar con validación cruzada o conjuntos de test, el escalado debe ajustarse **únicamente** con el conjunto de entrenamiento.

La forma idiomática y robusta de encapsular este flujo en Scikit-Learn es a través de un `Pipeline`:


In [ ]:
# Construccion del Pipeline de Produccion
pipe_knn = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=5, metric='minkowski', p=2))
])

# Entrenamiento completo en un solo paso
pipe_knn.fit(X_train, y_train)

# Evaluacion sobre test
y_pred_pipe = pipe_knn.predict(X_test)
print(classification_report(y_test, y_pred_pipe, target_names=cancer.target_names))


## 5. Conclusiones y Puntos Clave

1. **Lazy Learning**: KNN no aprende una función explícita durante `fit()`, simplemente memoriza los datos. Toda la computación ocurre en `predict()`.
2. **Métricas de Distancia**: La métrica por defecto es Minkowski con $p=2$ (Euclidiana), pero $p=1$ (Manhattan) resulta a menudo más robusta ante la presencia de variables correlacionadas o ruido.
3. **Escalado Obligatorio**: Nunca utilices KNN sobre características con diferentes unidades de medida sin aplicar previamente `StandardScaler` o `MinMaxScaler`.
4. **Data Leakage**: Empaqueta siempre el escalador y el clasificador dentro de un `Pipeline` para garantizar que la información de test o validación cruzada no contamine el preprocesamiento.

En el siguiente cuaderno, **02_Hiperparametros_Eleccion_K_y_Fronteras.ipynb**, estudiaremos a fondo el compromiso Sesgo-Varianza en función de $K$, esquemas de votación ponderada y visualización 2D de fronteras de decisión.
